# Shopee Review Classifier
## Analisis Sentimen Ulasan Pelanggan Shopee
Menggunakan TF-IDF dan Machine Learning (Multinomial Naive Bayes & Logistic Regression)

**Author:** Ida Bagus Adhiraga Yudhistira  
**Dataset:** Google Play Store, Shopee Indonesia  
**Kelas:** Positif, Netral, Negatif

## 0. Install Dependencies

In [ ]:
!pip install -q google-play-scraper PySastrawi emoji wordcloud

## 1. Import Library

In [ ]:
# Scraping
from google_play_scraper import reviews, Sort

# Data manipulation
import pandas as pd
import numpy as np

# Preprocessing
import re
import emoji
import nltk
from nltk.tokenize import word_tokenize
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory

# Modeling
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# Visualisasi
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud

# Setup NLTK
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
print('Libraries loaded.')

## 2. Scraping Ulasan Shopee dari Google Play Store

In [ ]:
result, _ = reviews(
    'com.shopee.id',
    lang='id',
    country='id',
    sort=Sort.NEWEST,
    count=2500
)

df = pd.DataFrame(result)[['content', 'score']]
df.columns = ['ulasan', 'rating']
df.dropna(inplace=True)
df.drop_duplicates(subset='ulasan', inplace=True)
df.reset_index(drop=True, inplace=True)

print(f'Total ulasan: {len(df)}')
df.head()

## 3. Eksplorasi Data (EDA)

In [ ]:
# Distribusi rating
print('Distribusi Rating:')
print(df['rating'].value_counts().sort_index())

# Panjang ulasan
df['panjang'] = df['ulasan'].apply(len)
print(f'\nPanjang ulasan rata-rata: {df["panjang"].mean():.0f} karakter')
print(f'Panjang ulasan min: {df["panjang"].min()}, max: {df["panjang"].max()}')

# Plot distribusi rating
plt.figure(figsize=(6, 4))
sns.countplot(x='rating', data=df, palette='Reds_r')
plt.title('Distribusi Rating Ulasan Shopee')
plt.xlabel('Rating (Bintang)')
plt.ylabel('Jumlah Ulasan')
plt.tight_layout()
plt.show()

## 4. Preprocessing Teks

In [ ]:
factory = StemmerFactory()
stemmer = factory.create_stemmer()

sw_factory = StopWordRemoverFactory()
stopword_list = sw_factory.get_stop_words()

def preprocess(text):
    text = emoji.replace_emoji(str(text), replace='')
    text = text.lower()
    text = re.sub(r'[^a-z\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    tokens = word_tokenize(text)
    tokens = [t for t in tokens if t not in stopword_list]
    tokens = [stemmer.stem(t) for t in tokens]
    return ' '.join(tokens)

df['ulasan_bersih'] = df['ulasan'].apply(preprocess)
print('Preprocessing selesai.')
df[['ulasan', 'ulasan_bersih']].head()

## 5. Labeling Otomatis Berdasarkan Rating

In [ ]:
def label_sentimen(rating):
    if rating >= 4:
        return 'Positif'
    elif rating == 3:
        return 'Netral'
    else:
        return 'Negatif'

df['label_sentimen'] = df['rating'].apply(label_sentimen)

print('Distribusi Label Sentimen:')
print(df['label_sentimen'].value_counts())

colors = {'Positif': '#437a22', 'Netral': '#d19900', 'Negatif': '#ee4d2d'}
df['label_sentimen'].value_counts().plot(kind='bar', color=[colors[l] for l in df['label_sentimen'].value_counts().index])
plt.title('Distribusi Kelas Sentimen')
plt.xlabel('Kelas')
plt.ylabel('Jumlah')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 6. TF-IDF Vectorization

In [ ]:
tfidf = TfidfVectorizer(max_features=3000)
X = tfidf.fit_transform(df['ulasan_bersih'])
y = df['label_sentimen']

print(f'Shape matriks TF-IDF: {X.shape}')

## 7. Split Data (80% Train, 20% Test)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Train: {X_train.shape[0]} sampel')
print(f'Test : {X_test.shape[0]} sampel')

## 8. Model 1 - Multinomial Naive Bayes

In [ ]:
nb = MultinomialNB()
nb.fit(X_train, y_train)
y_pred_nb = nb.predict(X_test)

acc_nb = accuracy_score(y_test, y_pred_nb)
print(f'Akurasi Naive Bayes: {acc_nb:.4f}')
print('\nClassification Report:')
print(classification_report(y_test, y_pred_nb))

cm_nb = confusion_matrix(y_test, y_pred_nb, labels=['Positif', 'Netral', 'Negatif'])
plt.figure(figsize=(6, 4))
sns.heatmap(cm_nb, annot=True, fmt='d', cmap='Oranges',
            xticklabels=['Positif', 'Netral', 'Negatif'],
            yticklabels=['Positif', 'Netral', 'Negatif'])
plt.title('Confusion Matrix - Naive Bayes')
plt.ylabel('Aktual'); plt.xlabel('Prediksi')
plt.tight_layout()
plt.show()

## 9. Model 2 - Logistic Regression

In [ ]:
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train, y_train)
y_pred_lr = lr.predict(X_test)

acc_lr = accuracy_score(y_test, y_pred_lr)
print(f'Akurasi Logistic Regression: {acc_lr:.4f}')
print('\nClassification Report:')
print(classification_report(y_test, y_pred_lr))

cm_lr = confusion_matrix(y_test, y_pred_lr, labels=['Positif', 'Netral', 'Negatif'])
plt.figure(figsize=(6, 4))
sns.heatmap(cm_lr, annot=True, fmt='d', cmap='Greens',
            xticklabels=['Positif', 'Netral', 'Negatif'],
            yticklabels=['Positif', 'Netral', 'Negatif'])
plt.title('Confusion Matrix - Logistic Regression')
plt.ylabel('Aktual'); plt.xlabel('Prediksi')
plt.tight_layout()
plt.show()

## 10. Perbandingan Akurasi Dua Model

In [ ]:
models = ['Naive Bayes', 'Logistic Regression']
accuracies = [acc_nb, acc_lr]

plt.figure(figsize=(6, 4))
bars = plt.bar(models, accuracies, color=['#d19900', '#437a22'], width=0.4)
plt.ylim(0, 1.05)
plt.title('Perbandingan Akurasi Model')
plt.ylabel('Akurasi')
for bar, acc in zip(bars, accuracies):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
             f'{acc:.2%}', ha='center', va='bottom', fontweight='bold')
plt.tight_layout()
plt.show()

## 11. WordCloud per Kelas Sentimen

In [ ]:
kelas_list = ['Positif', 'Netral', 'Negatif']
wc_colors = {'Positif': 'Greens', 'Netral': 'YlOrBr', 'Negatif': 'Reds'}

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, kelas in zip(axes, kelas_list):
    teks = ' '.join(df[df['label_sentimen'] == kelas]['ulasan_bersih'])
    wc = WordCloud(width=600, height=400, background_color='white',
                   colormap=wc_colors[kelas], max_words=100).generate(teks)
    ax.imshow(wc, interpolation='bilinear')
    ax.axis('off')
    ax.set_title(f'WordCloud - {kelas}', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.show()

## 12. Kata Dominan per Kelas (TF-IDF Score)

In [ ]:
top_n = 15
bar_colors = {'Positif': '#437a22', 'Netral': '#d19900', 'Negatif': '#ee4d2d'}

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, kelas in zip(axes, kelas_list):
    idx = df[df['label_sentimen'] == kelas].index
    tfidf_kelas = TfidfVectorizer(max_features=3000)
    X_kelas = tfidf_kelas.fit_transform(df.loc[idx, 'ulasan_bersih'])
    mean_scores = X_kelas.mean(axis=0).A1
    top_idx = mean_scores.argsort()[-top_n:][::-1]
    top_words = [tfidf_kelas.get_feature_names_out()[i] for i in top_idx]
    top_scores = mean_scores[top_idx]

    ax.barh(top_words[::-1], top_scores[::-1], color=bar_colors[kelas])
    ax.set_title(f'Top Kata - {kelas}', fontweight='bold')
    ax.set_xlabel('Rata-rata TF-IDF Score')

plt.tight_layout()
plt.show()

## 13. Kesimpulan dan Insight

In [ ]:
print('=== KESIMPULAN ===')
print(f'Akurasi Naive Bayes     : {acc_nb:.4f} ({"PASS" if acc_nb >= 0.75 else "BELUM PASS"})')
print(f'Akurasi Logistic Reg.   : {acc_lr:.4f} ({"PASS" if acc_lr >= 0.80 else "BELUM PASS"})')

best = 'Logistic Regression' if acc_lr > acc_nb else 'Naive Bayes'
print(f'\nModel terbaik: {best}')
print('\nLihat WordCloud dan grafik kata dominan untuk pola keluhan dan pujian pengguna Shopee.')